# 0. Problem
## 1934. Confirmation Rate — Medium
For each signup user, calculate confirmed requests / all confirmation requests. No requests => `0.00`. Round to 2 decimals.
Official: https://leetcode.com/problems/confirmation-rate/

# 1. Setup

In [ ]:
import pandas as pd
signups_rows=[(3,"2020-03-21 10:16:13"),(7,"2020-01-04 13:57:59"),(2,"2020-07-29 23:09:44"),(6,"2020-12-09 10:39:37")]
confirmations_rows=[(3,"2021-01-06","timeout"),(3,"2021-07-14","timeout"),(7,"2021-06-12","confirmed"),(7,"2021-06-13","confirmed"),(7,"2021-06-14","confirmed"),(2,"2021-01-22","confirmed"),(2,"2021-02-28","timeout")]
signups_pd=pd.DataFrame(signups_rows,columns=["user_id","time_stamp"])
confirmations_pd=pd.DataFrame(confirmations_rows,columns=["user_id","time_stamp","action"])

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark=SparkSession.builder.getOrCreate()
signups_spark=spark.createDataFrame(signups_rows,["user_id","time_stamp"])
confirmations_spark=spark.createDataFrame(confirmations_rows,["user_id","time_stamp","action"])
signups_spark.createOrReplaceTempView("Signups")
confirmations_spark.createOrReplaceTempView("Confirmations")

# 2. SQL Solution

In [ ]:
sql_result=spark.sql("""SELECT s.user_id,ROUND(AVG(CASE WHEN c.action='confirmed' THEN 1.0 ELSE 0.0 END),2) AS confirmation_rate FROM Signups s LEFT JOIN Confirmations c ON s.user_id=c.user_id GROUP BY s.user_id ORDER BY s.user_id""")
sql_result.show(truncate=False)

# 3. pandas Solution

In [ ]:
joined_pd=signups_pd[["user_id"]].merge(confirmations_pd[["user_id","action"]],on="user_id",how="left")
joined_pd["is_confirmed"]=joined_pd["action"].eq("confirmed").astype(float)
result_pd=(joined_pd.groupby("user_id",as_index=False).agg(confirmation_rate=("is_confirmed","mean")).assign(confirmation_rate=lambda d:d["confirmation_rate"].round(2)).sort_values("user_id").reset_index(drop=True))
result_pd

# 4. PySpark Solution

In [ ]:
result_spark=(signups_spark.join(confirmations_spark,on="user_id",how="left").groupBy("user_id").agg(F.round(F.avg(F.when(F.col("action")=="confirmed",1.0).otherwise(0.0)),2).alias("confirmation_rate")).orderBy("user_id"))
result_spark.show(truncate=False)

# 5. Pattern Mapping
| Concept | SQL | pandas | PySpark |
|---|---|---|---|
| conditional average | `AVG(CASE...)` | boolean→float→`.mean()` | `avg(when(...))` |
| preserve all users | `LEFT JOIN` | `how="left"` | `how="left"` |

# 6. Muscle-Memory Round

พิมพ์ใหม่เองโดยไม่ copy คำตอบด้านบน

In [ ]:
# MUSCLE MEMORY — SQL
# Rebuild using temp view(s): Signups, Confirmations

In [ ]:
# MUSCLE MEMORY — PANDAS
# Rebuild using: signups_pd, confirmations_pd

In [ ]:
# MUSCLE MEMORY — PYSPARK
# Rebuild using: signups_spark, confirmations_spark